# Astro `v2 final` 冻结表征 Redshift Probe 结果分析

这个 notebook 用来分析基于 **`v2 final checkpoint`** 的 frozen-backbone redshift probe 结果。

默认读取目录：`astro_sdss_v2_20gb_curated/probes/redshift_probe/`

主要输出：
- 各输入模式（`image-only` / `spectrum-only` / `both`）指标对比表
- `MAE / RMSE / outlier rate` 柱状图
- `z_true vs z_pred` 散点图
- 误差直方图
- `residual vs z_true`
- `residual vs sn_median_r`

这个 notebook **只做结果读取和可视化**，不负责训练。

In [ ]:
# Cell 0: 基础配置
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

PROJECT_ROOT = Path('/root/wyj/map-anything-sdss')
PROBE_ROOT = PROJECT_ROOT / 'astro_sdss_v2_20gb_curated' / 'probes' / 'redshift_probe'
CHECKPOINT_LABEL = 'checkpoint-final'
MODES = ['image-only', 'spectrum-only', 'both']

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'PROBE_ROOT   = {PROBE_ROOT}')
print(f'CHECKPOINT   = {CHECKPOINT_LABEL}')


In [ ]:
# Cell 1: 工具函数
CANONICAL_MODE_NAMES = {
    'image_only': 'image-only',
    'image-only': 'image-only',
    'spectrum_only': 'spectrum-only',
    'spectrum-only': 'spectrum-only',
    'both': 'both',
}


def canonical_mode_name(name: str) -> str:
    key = str(name).strip()
    return CANONICAL_MODE_NAMES.get(key, key)


def find_summary_csv(probe_root: Path) -> Path | None:
    direct = probe_root / 'summary.csv'
    if direct.is_file():
        return direct
    matches = sorted(probe_root.rglob('summary.csv'))
    return matches[0] if matches else None


def find_mode_dir(probe_root: Path, mode: str) -> Path | None:
    candidates = [
        probe_root / 'runs' / mode,
        probe_root / canonical_mode_name(mode),
        probe_root / mode,
    ]
    for candidate in candidates:
        if candidate.is_dir() and (candidate / 'metrics.json').is_file():
            return candidate
    for metrics_path in probe_root.rglob('metrics.json'):
        if canonical_mode_name(metrics_path.parent.name) == canonical_mode_name(mode):
            return metrics_path.parent
    return None


def safe_read_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def pick_first_existing(mapping: dict, keys: list[str], default=np.nan):
    for key in keys:
        if key in mapping:
            return mapping[key]
    return default


def pick_column(df: pd.DataFrame, candidates: list[str], required: bool = True):
    for column in candidates:
        if column in df.columns:
            return column
    if required:
        raise KeyError(f'Missing expected columns. candidates={candidates}, available={list(df.columns)}')
    return None


def load_mode_results(probe_root: Path, mode: str):
    mode_dir = find_mode_dir(probe_root, mode)
    if mode_dir is None:
        raise FileNotFoundError(f'Could not locate run directory for mode={mode} under {probe_root}')

    metrics_path = mode_dir / 'metrics.json'
    preds_path = mode_dir / 'predictions.csv'
    if not metrics_path.is_file():
        raise FileNotFoundError(metrics_path)
    if not preds_path.is_file():
        raise FileNotFoundError(preds_path)

    metrics = safe_read_json(metrics_path)
    preds = pd.read_csv(preds_path)

    z_true_col = pick_column(preds, ['z_true', 'target_z', 'z', 'y_true'])
    z_pred_col = pick_column(preds, ['z_pred', 'pred_z', 'prediction', 'y_pred'])
    sn_col = pick_column(preds, ['sn_median_r', 'snr_r', 'snr', 'sn'], required=False)

    out = preds.copy()
    out['mode'] = canonical_mode_name(mode)
    out['z_true'] = out[z_true_col].astype(float)
    out['z_pred'] = out[z_pred_col].astype(float)
    if sn_col is not None:
        out['sn_median_r'] = out[sn_col].astype(float)
    else:
        out['sn_median_r'] = np.nan
    out['delta_z'] = out['z_pred'] - out['z_true']
    out['abs_delta_over_1pz'] = np.abs(out['delta_z']) / (1.0 + out['z_true'])

    return mode_dir, metrics, out


def build_summary_table(probe_root: Path, modes: list[str]) -> pd.DataFrame:
    rows = []
    for mode in modes:
        mode_dir, metrics, preds = load_mode_results(probe_root, mode)
        rows.append({
            'mode': canonical_mode_name(mode),
            'mode_dir': str(mode_dir.relative_to(probe_root)),
            'num_samples': int(len(preds)),
            'mae_z': float(pick_first_existing(metrics, ['mae_z', 'val_mae_z', 'test_mae_z'])),
            'rmse_z': float(pick_first_existing(metrics, ['rmse_z', 'val_rmse_z', 'test_rmse_z'])),
            'median_abs_delta_over_1pz': float(pick_first_existing(metrics, ['median_abs_delta_over_1pz', 'median_abs_dz_over_1pz', 'median_delta_over_1pz'])),
            'catastrophic_outlier_rate': float(pick_first_existing(metrics, ['catastrophic_outlier_rate', 'outlier_rate', 'cat_outlier_rate'])),
            'pearson_corr': float(pick_first_existing(metrics, ['pearson_corr', 'corr', 'pearson_r'])),
            'spearman_corr': float(pick_first_existing(metrics, ['spearman_corr', 'rank_corr', 'spearman_rho'])),
        })
    summary = pd.DataFrame(rows).sort_values('mode').reset_index(drop=True)
    order = {'image-only': 0, 'spectrum-only': 1, 'both': 2}
    summary = summary.sort_values('mode', key=lambda s: s.map(order)).reset_index(drop=True)
    return summary


In [ ]:
# Cell 2: 读取 summary 与各 mode 结果
summary_path = find_summary_csv(PROBE_ROOT)
if summary_path is not None:
    print(f'Found summary.csv: {summary_path}')
    try:
        summary_csv = pd.read_csv(summary_path)
        display(summary_csv)
    except Exception as exc:
        print(f'Failed to read summary.csv: {exc}')
else:
    print('No summary.csv found; building summary from per-mode outputs instead.')

summary_df = build_summary_table(PROBE_ROOT, MODES)
display(summary_df)

mode_results = {}
for mode in MODES:
    mode_dir, metrics, preds = load_mode_results(PROBE_ROOT, mode)
    mode_results[canonical_mode_name(mode)] = {
        'dir': mode_dir,
        'metrics': metrics,
        'preds': preds,
    }

Markdown('### Loaded modes: ' + ', '.join(mode_results.keys()))


In [ ]:
# Cell 3: 核心指标对比柱状图
plot_df = summary_df.melt(
    id_vars=['mode'],
    value_vars=['mae_z', 'rmse_z', 'catastrophic_outlier_rate'],
    var_name='metric',
    value_name='value',
)

metric_titles = {
    'mae_z': 'MAE(z)',
    'rmse_z': 'RMSE(z)',
    'catastrophic_outlier_rate': 'Catastrophic Outlier Rate',
}
plot_df['metric_name'] = plot_df['metric'].map(metric_titles)

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(data=plot_df, x='metric_name', y='value', hue='mode', ax=ax)
ax.set_xlabel('')
ax.set_ylabel('Value')
ax.set_title(f'Redshift Probe Metrics Comparison ({CHECKPOINT_LABEL})')
ax.legend(title='mode')
plt.tight_layout()
plt.show()


In [ ]:
# Cell 4: 指标表（更适合论文草图对比）
display(
    summary_df.style.format({
        'mae_z': '{:.6f}',
        'rmse_z': '{:.6f}',
        'median_abs_delta_over_1pz': '{:.6f}',
        'catastrophic_outlier_rate': '{:.4%}',
        'pearson_corr': '{:.6f}',
        'spearman_corr': '{:.6f}',
    })
)


In [ ]:
# Cell 5: z_true vs z_pred 散点图
fig, axes = plt.subplots(1, len(MODES), figsize=(6 * len(MODES), 5), sharex=False, sharey=False)
if len(MODES) == 1:
    axes = [axes]

for ax, mode in zip(axes, MODES):
    result = mode_results[canonical_mode_name(mode)]
    preds = result['preds']
    ax.scatter(preds['z_true'], preds['z_pred'], s=12, alpha=0.45)
    z_min = float(np.nanmin([preds['z_true'].min(), preds['z_pred'].min()]))
    z_max = float(np.nanmax([preds['z_true'].max(), preds['z_pred'].max()]))
    ax.plot([z_min, z_max], [z_min, z_max], '--', color='black', linewidth=1.0)
    ax.set_title(canonical_mode_name(mode))
    ax.set_xlabel('z_true')
    ax.set_ylabel('z_pred')

fig.suptitle(f'z_true vs z_pred ({CHECKPOINT_LABEL})', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 6: 误差直方图 |Δz|/(1+z)
fig, ax = plt.subplots(figsize=(8.5, 5.5))
for mode in MODES:
    preds = mode_results[canonical_mode_name(mode)]['preds']
    sns.histplot(
        preds['abs_delta_over_1pz'],
        bins=40,
        stat='density',
        element='step',
        fill=False,
        common_norm=False,
        label=canonical_mode_name(mode),
        ax=ax,
    )
ax.set_xlabel(r'$|\Delta z| / (1 + z_{true})$')
ax.set_title(f'Error Distribution ({CHECKPOINT_LABEL})')
ax.legend(title='mode')
plt.tight_layout()
plt.show()


In [ ]:
# Cell 7: residual vs z_true
fig, axes = plt.subplots(1, len(MODES), figsize=(6 * len(MODES), 5), sharey=True)
if len(MODES) == 1:
    axes = [axes]

for ax, mode in zip(axes, MODES):
    preds = mode_results[canonical_mode_name(mode)]['preds']
    ax.scatter(preds['z_true'], preds['delta_z'], s=12, alpha=0.4)
    ax.axhline(0.0, linestyle='--', color='black', linewidth=1.0)
    ax.set_title(canonical_mode_name(mode))
    ax.set_xlabel('z_true')
    ax.set_ylabel('z_pred - z_true')

fig.suptitle(f'Residual vs z_true ({CHECKPOINT_LABEL})', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 8: residual vs sn_median_r
available_modes = [
    mode for mode in MODES
    if not mode_results[canonical_mode_name(mode)]['preds']['sn_median_r'].isna().all()
]

if not available_modes:
    print('`sn_median_r` column not found in predictions.csv; skip this panel.')
else:
    fig, axes = plt.subplots(1, len(available_modes), figsize=(6 * len(available_modes), 5), sharey=True)
    if len(available_modes) == 1:
        axes = [axes]

    for ax, mode in zip(axes, available_modes):
        preds = mode_results[canonical_mode_name(mode)]['preds']
        ax.scatter(preds['sn_median_r'], preds['abs_delta_over_1pz'], s=12, alpha=0.4)
        ax.set_title(canonical_mode_name(mode))
        ax.set_xlabel('sn_median_r')
        ax.set_ylabel(r'$|\Delta z| / (1 + z_{true})$')

    fig.suptitle(f'Error vs sn_median_r ({CHECKPOINT_LABEL})', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 9: 单独查看某个 mode 的原始 metrics / 头部预测样本
MODE_TO_INSPECT = 'both'
mode_key = canonical_mode_name(MODE_TO_INSPECT)
result = mode_results[mode_key]
print('mode dir:', result['dir'])
print('metrics:')
display(pd.Series(result['metrics']))
print('predictions head:')
display(result['preds'].head())
